In [ ]:
#Usando a biblioteca do Python

%pip -q install google-genai

In [ ]:
# Configura a API Key do Google Gemini

import os
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [ ]:
# Configura o cliente da SDK do Gemini

from google import genai
client = genai.Client()
MODEL_ID = "gemini-2.0-flash"

In [ ]:
# Instalar Framework de agentes do Google

!pip install -q google-adk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.1/232.1 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.1/217.1 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.1/334.1 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.8/65.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.0/119.0 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.9/194.9 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.

In [17]:
from google.adk.agents import Agent # Agente
from google.adk.runners import Runner # Orquestrador/Coordenador
from google.adk.sessions import InMemorySessionService # Memória interna do orquestrador
from google.adk.tools import google_search # Ferramenta do Google Search (para poder pesquisar)
from google.genai import types  # Para criar conteúdos (Content e Part) / fazer as configurações
from datetime import date
import textwrap # Para formatar melhor a saída de texto
from IPython.display import display, Markdown # Para exibir texto formatado no Colab
import requests # Para fazer requisições HTTP
import warnings

warnings.filterwarnings("ignore")

In [ ]:
# Função auxiliar que envia uma mensagem para um agente via Runner e retorna a resposta final
def call_agent(agent: Agent, message_text: str) -> str:
    # Cria um serviço de sessão em memória
    session_service = InMemorySessionService()
    # Cria uma nova sessão (você pode personalizar os IDs conforme necessário)
    session = session_service.create_session(app_name=agent.name, user_id="user1", session_id="session1")
    # Cria um Runner para o agente
    runner = Runner(agent=agent, app_name=agent.name, session_service=session_service)
    # Cria o conteúdo da mensagem de entrada
    content = types.Content(role="user", parts=[types.Part(text=message_text)])

    final_response = ""
    # Itera assincronamente pelos eventos retornados durante a execução do agente
    for event in runner.run(user_id="user1", session_id="session1", new_message=content):
        if event.is_final_response():
          for part in event.content.parts:
            if part.text is not None:
              final_response += part.text
              final_response += "\n"
    return final_response

In [ ]:
# Função auxiliar para exibir texto formatado em Markdown no Colab
def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

In [18]:
####################################################
# --- Agente 1: Analisador do perfil do usuário --- #
####################################################
def agente_analisador (info_usuario):
    analisador = Agent(
        name="agente_analisador",
        model="gemini-2.0-flash",
        instruction="""
        Você é o Agente Analisador de Respostas para um sistema de recomendação de animes.
        Sua tarefa é receber as respostas brutas do usuário (geradas por uma interação anterior) e transformá-las em um perfil de preferências
        estruturado e pronto para uso.

        As respostas do usuário são baseadas nas seguintes 5 perguntas:
        1. Quais são os 3 animes que você mais gostou de assistir até hoje e por quê?
        2. Quando você busca um novo anime, qual gênero ou estilo te atrai mais?
        3. O que você mais valoriza em um anime?
        4. Você prefere animes com uma trama fechada e poucos episódios, ou séries mais longas?
        5. Existe algum tema ou tipo de conteúdo que você prefere evitar em animes?

         Com base nessas respostas, você deve extrair e categorizar as informações-chave para formar um perfil de preferências.
         """,
         description="Responsável por coletar as preferências do usuário (estilo e humor) e definir um perfil de preferencias para a busca de animes.",
         tools=[google_search]  # Este agente não precisa de ferramentas externas
    )
    entrada_do_agente_analisador = f"""
    Informações do usuário:
    Melhores animes: {info_usuario.get('pergunta_1', '')}
    Gênero ou estilo de anime desejado: {info_usuario.get('pergunta_2', '')}
    O que mais atrai: {info_usuario.get('pergunta_3', '')}
    Duração: {info_usuario.get('pergunta_4', '')}
    Tema ou tipo de conteudo para evitar: {info_usuario.get('pergunta_5', '')}
    """
    resultados = call_agent(analisador, entrada_do_agente_analisador)
    return resultados

In [24]:
####################################################
# --- Agente 2: buscador --- #
####################################################
def agente_buscador(info_usuario, resultados_analisador):
  buscador = Agent(
    name="agente_buscador",
    model="gemini-2.0-flash",  # Substitua pelo modelo de linguagem que você está usando
    instruction="""
    Você é o Agente Buscador de Animes para um sistema de recomendação.
    Sua tarefa é encontrar animes que correspondam às preferências do usuário, utilizando as informações do perfil
    de preferências fornecido pelo Agente Analisador de Respostas.

    Sua busca deve ser orientada por:
    - Os gêneros e estilos de anime preferidos pelo usuário.
    - As características ou qualidades mais valorizadas em animes (ex: ação intensa, humor, personagens profundos, etc.).
    - A preferência de duração da série (animes curtos/trama fechada ou séries mais longas).
    - A exclusão rigorosa de animes com temas ou conteúdos a evitar.
    - A similaridade ou elementos em comum com os animes favoritos citados pelo usuário.

    Utilize sua base de dados do google (google_search) para identificar títulos que se encaixem nesses critérios.
    Selecione uma lista inicial de 5 a 10 animes promissores. Para cada um, colete o nome, seus gêneros, uma sinopse breve,
    e suas tags ou características principais.
    """,
    description="Utiliza o Google Search para encontrar animes com base no perfil fornecido pelo Agente Analisador.",
    tools=[google_search]
  )
  entrada_do_agente_buscador = f"""
  Análise do Agente Analisador: {resultados_analisador}
  Melhores animes: {info_usuario.get('pergunta_1', '')}
  Gênero ou estilo de anime desejado: {info_usuario.get('pergunta_2', '')}
  O que mais atrai: {info_usuario.get('pergunta_3', '')}
  Duração: {info_usuario.get('pergunta_4', '')}
  Tema ou tipo de conteúdo para evitar: {info_usuario.get('pergunta_5', '')}
  """

  animes_encontrados = call_agent(buscador, entrada_do_agente_buscador)
  return animes_encontrados




In [20]:
####################################################
# --- Agente 3: Revisor --- #
####################################################
def agente_revisor(info_usuario, animes_encontrados, resultados_analisador):
  revisor = Agent(
    name="agente_sugestao",
    model="gemini-2.0-flash",
    instruction="""
    Você é um Editor e Revisor de Recomendações de Animes, especialista em entender as nuances das preferências de otaku e em garantir a qualidade das sugestões.
    Sua tarefa é avaliar criticamente os animes sugeridos pelo Agente Buscador, assegurando a máxima compatibilidade com o perfil de preferências do usuário.

    Com base no perfil de preferências do usuário (recebido do Agente Analisador de Respostas) e na lista de animes encontrados pelo Agente Buscador, você deve
    analisar cada anime individualmente, considerando os seguintes aspectos:

    - Alinhamento com Gêneros e Estilos: O anime corresponde aos gêneros ou estilos de anime que o usuário prefere (ex: Ação, Fantasia, Comédia, Romance)?
    Há alguma sugestão que se desvie significativamente desses gostos?
    - Sintonia com Características Valorizadas: O anime oferece as qualidades que o usuário mais valoriza (ex: histórias complexas, personagens cativantes,
    humor leve, muita ação)? As características do anime refletem esses valores?
    - Respeito à Duração e Temas a Evitar: O anime está de acordo com a preferência de duração do usuário (curto ou longo)? E, crucialmente, ele **NÃO**
    contém nenhum dos temas ou conteúdos que o usuário expressamente pediu para evitar (ex: violência extrema, temas infantis)?
    - Conexão com Animes de Referência: Há uma relação clara e justificável entre o anime sugerido e os animes favoritos que o usuário já citou? Essa conexão
    é lógica e faz sentido para uma recomendação?

    Se todos os 3 a 5 animes selecionados pelo Buscador atenderem satisfatoriamente a todos os critérios de compatibilidade e qualidade, sua resposta final deve
    ser uma lista concisa e bem justificada dessas recomendações, cada uma com o nome do anime, uma sinopse breve e um motivo claro que explique por que o anime
    é uma excelente escolha para o usuário.

    Caso encontre problemas em alguma sugestão ou na lista como um todo, seja específico ao apontá-los. Sugira melhorias concretas, como a remoção de animes que
    não se encaixam, a busca por alternativas mais alinhadas, ou ajustes nas justificativas de recomendação para torná-las mais persuasivas e precisas.
    """,
    description="Garante que os animes recomendados correspondam perfeitamente ao perfil apresentado pelo usuário, oferecendo apenas o melhor.",
    tools=[google_search]
  )
  entrada_do_agente_revisor = f"""
  Análise do Agente Analisador: {resultados_analisador}
  Melhores animes: {info_usuario.get('pergunta_1', '')}
  Gênero ou estilo de anime desejado: {info_usuario.get('pergunta_2', '')}
  O que mais atrai: {info_usuario.get('pergunta_3', '')}
  Duração: {info_usuario.get('pergunta_4', '')}
  Tema ou tipo de conteudo para evitar: {info_usuario.get('pergunta_5', '')}
  Animes encontrados: {animes_encontrados}
    """

  animes_revisados = call_agent(revisor, entrada_do_agente_revisor)
  return animes_revisados

In [22]:
### comeca a interagir com o usuario

print("Não sabe que animes assistir hoje? Vem que esses 3 agentes vão lhe ajudar! 🎬 😊")
print("\n")
print("Primeiramente: precisamos conhecer um pouco sobre você e o que você gosta de assistir!")
print("\n")

# --- Obter as Preferências de Anime e Humor do Usuário ---
info_usuario = {}
cont = 0
num_perguntas = 5

perguntas_usuario = [
    "Quais são os 3 animes que você mais gostou de assistir até hoje e por quê?",
    "Quando você busca um novo anime, qual gênero ou estilo te atrai mais?",
    "O que você mais valoriza em um anime? (historia, personagens complexos, cenario, ação...)",
    "Você prefere animes com uma trama fechada e poucos episódios, ou séries mais longas com a história se desenvolvendo ao longo do tempo?",
    "Existe algum tema ou tipo de conteúdo que você prefere evitar em animes?"
]

# Criando um chat bot curto para o primeiro agente ter as informações que precisa
while cont < num_perguntas:
    resposta = input(f"{perguntas_usuario[cont]} ")
    info_usuario[f"pergunta_{cont+1}"] = resposta
    cont += 1

print("\nObrigado pelas informações!")
print("Suas respostas:", info_usuario)
print("\n")

# --- Próximos passos (envio para os agentes) ---

Não sabe que animes assistir hoje? Vem que esses 3 agentes vão lhe ajudar! 🎬 😊


Primeiramente: precisamos conhecer um pouco sobre você e o que você gosta de assistir!


Quais são os 3 animes que você mais gostou de assistir até hoje e por quê? attack on titan, demons slayer, jujutsu no kaisen
Quando você busca um novo anime, qual gênero ou estilo te atrai mais? gosto de ação
O que você mais valoriza em um anime? (historia, personagens complexos, cenario, ação...) personagens complexos
Você prefere animes com uma trama fechada e poucos episódios, ou séries mais longas com a história se desenvolvendo ao longo do tempo? gosto de tudo
Existe algum tema ou tipo de conteúdo que você prefere evitar em animes? não gosto de romance

Obrigado pelas informações!
Suas respostas: {'pergunta_1': 'attack on titan, demons slayer, jujutsu no kaisen', 'pergunta_2': 'gosto de ação', 'pergunta_3': 'personagens complexos', 'pergunta_4': 'gosto de tudo', 'pergunta_5': 'não gosto de romance'}




In [25]:
# --- Próximos passos (envio para os agentes) ---

# --- Próximos passos (envio para os agentes) ---

# --- Executar os agentes ---

print(f"Maravilha! Vamos agora descobrir seus próximos animes!")


resultados_analisador = agente_analisador(info_usuario)
print("\n--- 📝 Resultado do Agente 1 (Analisador) ---\n")
display(to_markdown(resultados_analisador))
print("--------------------------------------------------------------")


animes_encontrados = agente_buscador(info_usuario, resultados_analisador)
print("\n--- 📝 Resultado do Agente 2 (Buscador) ---\n")
display(to_markdown(animes_encontrados))
print("--------------------------------------------------------------")


animes_revisados = agente_revisor(info_usuario, animes_encontrados, resultados_analisador)
print("\n--- 📝 Resultado do Agente 3 (Revisor) ---\n")
display(to_markdown(animes_revisados))
print("--------------------------------------------------------------")

Maravilha! Vamos agora descobrir seus próximos animes!

--- 📝 Resultado do Agente 1 (Analisador) ---



> Para criar um perfil de preferências de anime com base nas suas respostas, organizei as informações da seguinte forma:
> 
> *   **Animes Favoritos:**
>     *   Attack on Titan
>     *   Demon Slayer
>     *   Jujutsu Kaisen
> *   **Gênero/Estilo:** Ação
> *   **Atributos Valorizados:** Personagens complexos
> *   **Duração Preferida:** Sem preferência (gosta de todos os tipos de duração)
> *   **Temas a Evitar:** Romance
> 


--------------------------------------------------------------

--- 📝 Resultado do Agente 2 (Buscador) ---



> Certo, com base nas suas preferências, vou buscar animes que combinem ação, personagens complexos e que evitem o romance, priorizando aqueles que se assemelhem a "Attack on Titan", "Demon Slayer" e "Jujutsu Kaisen".
> 
> 
> Com base nas suas preferências e nas informações coletadas, aqui estão algumas sugestões de animes que combinam ação, personagens complexos e evitam o romance, lembrando que a ausência completa de elementos românticos é difícil de garantir, mas priorizei aqueles com foco mínimo nesse aspecto:
> 
> 1.  **Fullmetal Alchemist: Brotherhood:**
> 
>     *   **Gêneros:** Ação, Aventura, Fantasia, Militar, Shounen.
>     *   **Sinopse:** Acompanha os irmãos Edward e Alphonse Elric em busca da Pedra Filosofal para restaurar seus corpos após uma tentativa fracassada de reviver sua mãe através da alquimia.
>     *   **Características:** Personagens complexos com dilemas morais, sistema de magia interessante, ação constante e uma trama envolvente com reviravoltas.
> 2.  **Vinland Saga:**
> 
>     *   **Gêneros:** Ação, Aventura, Drama, Histórico, Seinen.
>     *   **Sinopse:** Situado na era Viking, segue a jornada de Thorfinn em busca de vingança pela morte de seu pai, ao mesmo tempo em que explora temas de violência, redenção e o significado da vida.
>     *   **Características:** Personagens complexos com desenvolvimento profundo, ação brutal e realista, ambientação histórica rica e uma narrativa que questiona os valores da guerra e da vingança.
> 3.  **Chainsaw Man:**
> 
>     *   **Gêneros:** Ação, Demônios, Horror, Shounen, Sobrenatural.
>     *   **Sinopse:** Denji, um jovem pobre que faz um pacto com um demônio motosserra para sobreviver, junta-se a um grupo de caçadores de demônios para proteger o mundo de ameaças sobrenaturais.
>     *   **Características:** Ação intensa e sangrenta, personagens caóticos e moralmente ambíguos, humor ácido e uma trama imprevisível com elementos de horror.
> 4.  **Psycho-Pass:**
> 
>     *   **Gêneros:** Ação, Ficção Científica, Policial, Psicológico, Seinen.
>     *   **Sinopse:** Em um futuro distópico onde o "Psycho-Pass" de cada indivíduo é constantemente monitorado para detectar tendências criminosas, a história segue Akane Tsunemori, uma inspetora novata que questiona a justiça do sistema.
>     *   **Características:** Personagens complexos com dilemas éticos, mundo distópico bem construído, ação estratégica e uma trama que explora temas de livre arbítrio, justiça e o papel da tecnologia na sociedade.
> 5.  **Code Geass:**
> 
>     *   **Gêneros:** Ação, Militar, Ficção Científica, Superpoderes, Suspense.
>     *   **Sinopse:** Em um mundo onde o Sacro Império da Britannia conquistou o Japão, Lelouch Lamperouge, um estudante exilado, recebe o poder "Geass" e lidera uma rebelião contra o império opressor.
>     *   **Características:** Personagens complexos com ideais conflitantes, ação estratégica com mechas, reviravoltas na trama e uma narrativa que explora temas de moralidade, poder e revolução.
> 6.  **Dorohedoro:**
> 
>     *   **Gêneros:** Ação, Aventura, Comédia, Fantasia, Horror, Seinen.
>     *   **Sinopse:** Em um mundo onde feiticeiros usam humanos como cobaias para seus experimentos, Caiman, um homem com cabeça de lagarto em busca de sua verdadeira identidade, caça os feiticeiros responsáveis por sua transformação.
>     *   **Características:** Mundo bizarro e original, personagens excêntricos e carismáticos, ação violenta e bem coreografada, e uma trama que mistura humor negro com elementos de horror e mistério.
> 7.  **Bungou Stray Dogs:**
> 
>     *   **Gêneros:** Ação, Mistério, Sobrenatural, Seinen.
>     *   **Sinopse:** Acompanha Atsushi Nakajima, um órfão com a habilidade de se transformar em um tigre, enquanto ele se junta à Agência de Detetives Armados, um grupo de detetives com habilidades sobrenaturais que resolvem casos perigosos.
>     *   **Características:** Personagens com habilidades únicas baseadas em famosos autores e suas obras, ação com elementos sobrenaturais, mistério e uma trama que explora temas de identidade, redenção e o poder da literatura.
> 
> Espero que essas sugestões sejam do seu agrado! Se tiver mais alguma preferência ou detalhe que queira adicionar, me diga para que eu possa refinar ainda mais a busca.
> 


--------------------------------------------------------------

--- 📝 Resultado do Agente 3 (Revisor) ---



> A lista de animes sugerida pelo Agente Buscador é muito boa e alinhada com as preferências do usuário, mas podemos otimizar ainda mais as justificativas e eliminar uma sugestão que pode não ser tão interessante.
> 
> **Animes Recomendados:**
> 
> 1.  **Fullmetal Alchemist: Brotherhood:**
>     *   **Gêneros:** Ação, Aventura, Fantasia, Militar, Shounen.
>     *   **Sinopse:** Acompanha os irmãos Edward e Alphonse Elric em busca da Pedra Filosofal para restaurar seus corpos após uma tentativa fracassada de reviver sua mãe através da alquimia.
>     *   **Motivo:** Este anime é uma excelente escolha devido à sua ação constante, personagens com dilemas morais complexos e um sistema de magia interessante. A trama envolvente e cheia de reviravoltas mantém o espectador engajado do início ao fim, similar à intensidade encontrada em "Attack on Titan" e "Demon Slayer".
> 
> 2.  **Vinland Saga:**
>     *   **Gêneros:** Ação, Aventura, Drama, Histórico, Seinen.
>     *   **Sinopse:** Situado na era Viking, segue a jornada de Thorfinn em busca de vingança pela morte de seu pai, ao mesmo tempo em que explora temas de violência, redenção e o significado da vida.
>     *   **Motivo:** "Vinland Saga" oferece personagens complexos com desenvolvimento profundo e ação brutal e realista. A ambientação histórica rica e a narrativa que questiona os valores da guerra e da vingança proporcionam uma experiência intensa e reflexiva, comparável à profundidade temática de "Attack on Titan".
> 
> 3.  **Chainsaw Man:**
>     *   **Gêneros:** Ação, Demônios, Horror, Shounen, Sobrenatural.
>     *   **Sinopse:** Denji, um jovem pobre que faz um pacto com um demônio motosserra para sobreviver, junta-se a um grupo de caçadores de demônios para proteger o mundo de ameaças sobrenaturais.
>     *   **Motivo:** "Chainsaw Man" é recomendado pela sua ação intensa e sangrenta, personagens caóticos e moralmente ambíguos, e humor ácido. A trama imprevisível com elementos de horror mantém o espectador na ponta da cadeira, assim como os momentos mais chocantes de "Jujutsu Kaisen".
> 
> 4.  **Psycho-Pass:**
>     *   **Gêneros:** Ação, Ficção Científica, Policial, Psicológico, Seinen.
>     *   **Sinopse:** Em um futuro distópico onde o "Psycho-Pass" de cada indivíduo é constantemente monitorado para detectar tendências criminosas, a história segue Akane Tsunemori, uma inspetora novata que questiona a justiça do sistema.
>     *   **Motivo:** Este anime é recomendado por seus personagens complexos com dilemas éticos e um mundo distópico bem construído. A ação estratégica e a trama que explora temas de livre arbítrio, justiça e o papel da tecnologia na sociedade oferecem uma experiência intelectualmente estimulante, similar à complexidade de "Attack on Titan".
> 
> 5.  **Code Geass:**
>     *   **Gêneros:** Ação, Militar, Ficção Científica, Superpoderes, Suspense.
>     *   **Sinopse:** Em um mundo onde o Sacro Império da Britannia conquistou o Japão, Lelouch Lamperouge, um estudante exilado, recebe o poder "Geass" e lidera uma rebelião contra o império opressor.
>     *   **Motivo:** "Code Geass" é uma excelente escolha devido aos seus personagens complexos com ideais conflitantes e ação estratégica com mechas. As reviravoltas na trama e a narrativa que explora temas de moralidade, poder e revolução proporcionam uma experiência rica e envolvente, semelhante à intensidade de "Attack on Titan".
> 
> **Anime Removido e Justificativa:**
> 
> *   **Bungou Stray Dogs:** Embora possua ação e personagens com habilidades únicas, a temática focada em literatura pode não ressoar tão fortemente com o usuário que busca ação intensa e personagens complexos. Além disso, o foco em mistério pode diluir o impacto da ação, tornando-o menos atraente em comparação com os outros animes da lista.
> 
> **Observações:**
> 
> *   A lista foi refinada para garantir que cada anime ofereça uma experiência que se alinhe fortemente com as preferências do usuário, evitando temas que possam não ser de seu interesse.
> *   As justificativas foram aprimoradas para destacar as qualidades que tornam cada anime uma excelente escolha, conectando-os diretamente aos animes favoritos do usuário.


--------------------------------------------------------------
